A retail company receives daily sales transaction files from multiple store locations in an Azure Data Lake folder. Instead of reprocessing all historical data every day, the data engineering team uses Spark Structured Streaming to incrementally load only the newly arrived files into a Delta table. This ensures timely updates to analytics dashboards while optimizing compute costs and processing time.

## **Streaming Query**

In [0]:
my_schema = """
    order_id INT,
    customer_id INT,
    order_date DATE,
    amount DOUBLE
"""

In [0]:
df_batch = spark.read.format("csv")\
    .option("header","true")\
    .schema(my_schema)\
    .load("/Volumes/pyspark_cata/source/db_volume/streamSource/")

display(df_batch)

In [0]:
df = spark.readStream.format("csv")\
    .option("header","true")\
    .schema(my_schema)\
    .load("/Volumes/pyspark_cata/source/db_volume/streamSource/")

## **Streaming Output**

In [0]:
df.writeStream.format("delta")\
  .option("checkpointLocation","/Volumes/pyspark_cata/source/db_volume/streamSink/checkpoint")\
  .option("mergeSchema", True)\
  .trigger(once=True)\
  .start("/Volumes/pyspark_cata/source/db_volume/streamSink/data")

In [0]:
%sql
SELECT * FROM delta.`/Volumes/pyspark_cata/source/db_volume/streamSink/data/`